# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is published via a Croissant schema URL and contains multiple record sets and fields related to protein mass spectrometry analysis on extracellular vesicles from human mast cells.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and instantiate the dataset object using `mlcroissant`. The URL is provided for convenience.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for this dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Access metadata object
meta = dataset.metadata

# Print dataset title and description
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
List available record sets, their `@id`s, and summarize the fields within each record set.

We'll iterate through the dataset's record sets, referencing each entity by its `@id` as per Croissant best practices.

In [ ]:
# Show a summary of record sets and their fields in the dataset
record_sets = list(dataset.record_sets)

print(f"Total record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record Set '@id': {rs['@id']}")
    record_set_ids.append(rs['@id'])
    # List the fields for this record set
    fields = rs.get('fields', [])
    print(f"  Number of fields: {len(fields)}")
    if fields:
        print("  Fields (by @id):")
        for f in fields:
            print(f"    - {f['@id']} (name: {f.get('name', f['@id'])})")
    print()

## 3. Data Extraction
Load all records from each record set to pandas DataFrames for further exploration. Use the `@id` of each record set for iteration and field references.

_Note: You can select a specific record set for analysis by its `@id` shown above. There may be more than one record set, e.g., for protein abundance, modifications, peptides, etc._

In [ ]:
# Extract data from each record set into a pandas DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Shape: {df.shape}, Columns: {list(df.columns)}\n")

# For demonstration, select the first record set
main_record_set_id = record_set_ids[0]
# Show a sample
print(f"Sample rows from record set {main_record_set_id}:")
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA using one of the numeric fields from the main record set. We'll reference fields by their `@id`s, and illustrate how to filter, normalize, and group data.

In [ ]:
# Identify numeric fields in the main record set
df = dataframes[main_record_set_id]
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields in {main_record_set_id}: {numeric_columns}\n")

# For demonstration, select the first numeric field
if numeric_columns:
    numeric_field = numeric_columns[0]  # e.g., 'abundance' or 'MW' if present
else:
    print("No numeric fields detected. Please check the schema or data.")
    numeric_field = None

# Example: Filter outliers where the numeric field value > threshold
if numeric_field:
    threshold = df[numeric_field].quantile(0.95)  # filter top 5% outliers as example
    filtered_df = df[df[numeric_field] < threshold]
    print(f"Filtered records in '{main_record_set_id}' with {numeric_field} < {threshold:.2f} (below 95th percentile): {len(filtered_df)} rows\n")

    # Normalize the numeric column
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Top normalized values in '{numeric_field}':")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a relevant categorical field (e.g., 'description' or another field)
    group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by '{group_field}':")
        display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the grouped means (if computed).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f'Distribution of {numeric_field} in Record Set {main_record_set_id}')
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    # If grouping performed show top groups
    if 'grouped_df' in locals():
        plt.figure(figsize=(12, 5))
        top_n = 15  # show top 15 groups
        sns.barplot(
            data=grouped_df.sort_values(numeric_field, ascending=False).head(top_n),
            x=numeric_field, y=group_field
        )
        plt.title(f'Top {top_n} {group_field} by Mean {numeric_field}')
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR^2 dataset describing protein abundance in human extracellular vesicles using `mlcroissant`, explored its schema and records via `@id`s, and carried out simple filtering and visualizations. This notebook provides a template for deeper analysis of Croissant-compliant datasets relevant to proteomics and biomedicine.

For further analysis, consider investigating sequence motifs, modification types, and abundance relationships across biological samples as referenced by `@id` within the schema.